<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/cournot_3d_nonpotential_stochastic_mlp_dtb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D non-potential Cournot: stochastic neural-DTB with score evolution

## Three-player game setup

For player $i$, let $r_i=\sum_{j\ne i}x_j$. With $b=1$ and $\mu=2$, the
payoff-gradient velocity from the thesis is

$$
b_i(x)=-2x_i+4r_i-4r_i^2.
$$

The stochastic dynamics use three independent Brownian motions with the exact
thesis amplitudes

$$
\sigma_1=\sigma_2=\sigma_3=0.1,
\qquad D=\operatorname{diag}(\sigma_1^2,\sigma_2^2,\sigma_3^2)=0.01I_3.
$$

The initial distribution is uniform on $[0,1]^3$, matching the thesis
non-potential experiment. The five reported reference equilibria are the
origin, $(3/8,3/8,3/8)$, and the three permutations of $(1/2,1/2,0)$.
The origin is unstable and the other four are reported as stable. This
notebook compares stochastic neural-DTB with a matched Euler--Maruyama (EM)
baseline. Both methods start from the same particles and use the same time grid
and noise amplitudes. The labeled references are zeros of the deterministic
drift implemented here; they are not point equilibria of the noisy trajectories.


## Block 0 — Shared repository setup

This notebook reuses `ResidualMLPMap`, `game_dtb_basis_matrices`, and
`map_at` from `run_game_dtb.py`; `count_trainable` from `network.py`;
and `flat_params` plus `jform_solve` from `dtb.py`. Spatial tangent
derivatives and score evolution come from the shared `utility.py`.

In [ ]:
# BLOCK 0 — Locate the shared modules locally or clone the requested branch in Colab.
from pathlib import Path
import math
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

MODULE_FILES = ('run_game_dtb.py', 'network.py', 'dtb.py', 'utility.py')
EXPERIMENT_DIR = next(
    (
        folder
        for folder in (Path.cwd(), Path.cwd() / 'DTB_Game_Ver2')
        if all((folder / name).is_file() for name in MODULE_FILES)
    ),
    None,
)

if EXPERIMENT_DIR is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run from the repository root or DTB_Game_Ver2 directory.')
    repo = Path('/content/dtb-colab-experiments')
    if not (repo / '.git').exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo),
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(repo), 'checkout', 'codex/game-dynamics-dtb'], check=True)
        subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
    EXPERIMENT_DIR = repo / 'DTB_Game_Ver2'

# Import the existing DTB, network, and map primitives requested for reuse.
sys.path.insert(0, str(EXPERIMENT_DIR.resolve()))
from run_game_dtb import ResidualMLPMap, game_dtb_basis_matrices
from network import count_trainable
from dtb import device, flat_params, jform_solve
from importlib import reload
import utility as stochastic_utility

stochastic_utility = reload(stochastic_utility)
from utility import (
    create_run_directory,
    diagnostics_markdown,
    euler_score_update,
    plot_tangent_diagnostics,
    sample_initial_with_score,
    save_run_data,
    tangent_velocity_spatial_terms,
)

print('Shared module directory:', EXPERIMENT_DIR)

## Block 1 — Game and experiment controls

All numerical, neural-network, stochastic, saving, and visualization
parameters are editable below. The complete realized configuration is
printed after the particles and tangent basis have been initialized.

In [ ]:
# BLOCK 1 — Define the three-player game and every editable experiment control.
SAVE_RUN = False  # True creates a uniquely named folder under saved_runs/.
SEED = 2026
EM_SEED = SEED + 10_000

DIM = 3
N_PARTICLES = 2000
H = 0.005
T_FINAL = 1.0
N_STEPS = round(T_FINAL / H)
SNAPSHOT_TIMES = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)

# The paper uses independent noise of amplitude 0.1 in every coordinate.
NOISE_STD = (0.1, 0.1, 0.1)

# "uniform" matches the paper's samples and uses the exact interior score q_0=0.
# "gaussian" and "smoothed_uniform" provide globally smooth analytical scores.
INITIAL_LAW = 'uniform'
GAUSSIAN_MEAN = 0.5
GAUSSIAN_STD = 0.15
UNIFORM_SMOOTHING_STD = 0.02

NN_CHOICE = 'mlp'
NN_ACTIVATION = 'tanh'  # Smooth activation is required by grad(div(u)).
MLP_WIDTH = 32
MLP_DEPTH = 4
BASIS_SIZE = 128
SVD_RTOL = 1e-3
SVD_METHOD = 'svd_gpu'
JACOBIAN_CHUNK = 64
DERIVATIVE_CHUNK = 32
PRINT_EVERY = 20

DTYPE = torch.float32
DEVICE = device()
COURNOT_B = 1.0
COURNOT_MU = 2.0
COORDINATE_PAIRS = ((1, 2),)

# Separate 3D figures use the same view, limits, and displayed particle labels.
PLOT_3D_COLUMNS = 2
PLOT_3D_MAX_POINTS = None  # None shows every particle; otherwise a seeded subset.
PLOT_3D_ELEV = 22.0
PLOT_3D_AZIM = -55.0
PLOT_3D_POINT_SIZE = 2.0
PLOT_3D_ALPHA = 0.35

def game_velocity(x):
    """Three-player non-potential Cournot payoff-gradient field."""
    if x.shape[-1] != DIM:
        raise ValueError(f'Expected states with last dimension {DIM}.')
    rivals = x.sum(dim=-1, keepdim=True) - x
    return -2.0 * COURNOT_B * x + 2.0 * COURNOT_B * COURNOT_MU * rivals * (1.0 - rivals)

KNOWN_EQUILIBRIA = torch.tensor([
    [0.0, 0.0, 0.0],
    [3/8, 3/8, 3/8],
    [1/2, 1/2, 0.0],
    [1/2, 0.0, 1/2],
    [0.0, 1/2, 1/2],
], device=DEVICE, dtype=DTYPE)
STABLE_MASK = np.array([False, True, True, True, True])
EQUILIBRIUM_LABELS = [f'E_{index}' for index in range(len(KNOWN_EQUILIBRIA))]

## Block 2 — Particle, score, and neural tangent initialization

The persistent labels start at $x_i^0=X_0(z_i)=z_i$. Each label carries
its score $q_i^0=\nabla\log\rho_0(x_i^0)$. A single seeded subset of MLP
parameter directions defines the tangent basis for the complete run.

In [ ]:
# BLOCK 2 — Initialize particles, their scores, and the fixed neural tangent basis.
if NN_ACTIVATION not in {'tanh', 'gelu', 'silu'}:
    raise ValueError('Score evolution requires a twice-differentiable activation.')
if H <= 0 or N_STEPS < 1:
    raise ValueError('H and N_STEPS must be positive.')
if not np.isclose(N_STEPS * H, T_FINAL):
    raise ValueError('T_FINAL must be an integer multiple of H.')
if len(NOISE_STD) != DIM:
    raise ValueError('NOISE_STD needs one independent amplitude per player.')

# Seed the network and use independent recorded streams for particles and EM noise.
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
generator_device = DEVICE.type if DEVICE.type == 'cuda' else 'cpu'
initial_generator = torch.Generator(device=generator_device).manual_seed(SEED)
em_generator = torch.Generator(device=generator_device).manual_seed(EM_SEED)

# Attach the analytical initial score to every persistent particle label.
x_0, q_0, log_density_0 = sample_initial_with_score(
    N_PARTICLES,
    DIM,
    law=INITIAL_LAW,
    device=DEVICE,
    dtype=DTYPE,
    generator=initial_generator,
    gaussian_mean=GAUSSIAN_MEAN,
    gaussian_std=GAUSSIAN_STD,
    smoothing_std=UNIFORM_SMOOTHING_STD,
)
x_k = x_0.clone()
q_k = q_0.clone()
log_density_k = log_density_0.clone()
em_k = x_0.clone()  # Exactly the same initial point cloud as DTB.

# The fixed MLP provides tangent directions; it is not trained during the run.
mlp = ResidualMLPMap(
    dim=DIM,
    width=MLP_WIDTH,
    depth=MLP_DEPTH,
    activation=NN_ACTIVATION,
    dtype=DTYPE,
    zero_init_output=False,
).net.to(DEVICE)
N_PARAMETERS = count_trainable(mlp)
theta_0, structure, _ = flat_params(mlp)
mlp.requires_grad_(False)

# Select one reproducible random sub-basis and keep it fixed for all time steps.
if not isinstance(BASIS_SIZE, int) or not 1 <= BASIS_SIZE <= N_PARAMETERS:
    raise ValueError(f'BASIS_SIZE must be in [1, {N_PARAMETERS}].')
basis_generator = torch.Generator(device='cpu').manual_seed(SEED)
selected = torch.randperm(N_PARAMETERS, generator=basis_generator)[:BASIS_SIZE]
selected = selected.sort().values.to(DEVICE)

# Sigma is the SDE amplitude; D=Sigma Sigma^T is the Fokker--Planck covariance.
sigma = torch.tensor(NOISE_STD, device=DEVICE, dtype=DTYPE)
diffusion = torch.diag(sigma.square())

# Verify that every plotted analytical reference is stationary for this game field.
equilibrium_residual = float(game_velocity(KNOWN_EQUILIBRIA).norm(dim=1).max())
if equilibrium_residual > 2e-5:
    raise ValueError(f'An equilibrium reference has drift residual {equilibrium_residual:.3e}.')

# Convert requested physical times to exact stored state indices.
snapshot_steps = [round(value / H) for value in SNAPSHOT_TIMES]
if not snapshot_steps or any(step < 0 or step > N_STEPS for step in snapshot_steps):
    raise ValueError('Snapshot times must be nonempty and lie in [0, T_FINAL].')
if any(not np.isclose(step * H, value) for step, value in zip(snapshot_steps, SNAPSHOT_TIMES)):
    raise ValueError('Every snapshot time must lie on the time grid.')

# Save every experiment setting and the exact selected parameter indices.
CONFIG = {
    'game_dimension': DIM,
    'particles': N_PARTICLES,
    'step_size': H,
    'steps': N_STEPS,
    'final_time': T_FINAL,
    'snapshot_times': list(SNAPSHOT_TIMES),
    'noise_std': list(NOISE_STD),
    'diffusion_diagonal': sigma.square().detach().cpu().tolist(),
    'maximum_equilibrium_residual': equilibrium_residual,
    'initial_law': INITIAL_LAW,
    'gaussian_mean': GAUSSIAN_MEAN,
    'gaussian_std': GAUSSIAN_STD,
    'uniform_smoothing_std': UNIFORM_SMOOTHING_STD,
    'network': NN_CHOICE,
    'activation': NN_ACTIVATION,
    'width': MLP_WIDTH,
    'depth': MLP_DEPTH,
    'trainable_parameters': N_PARAMETERS,
    'basis_size': BASIS_SIZE,
    'selected_parameter_indices': selected.detach().cpu().tolist(),
    'svd_rtol': SVD_RTOL,
    'svd_method': SVD_METHOD,
    'jacobian_chunk': JACOBIAN_CHUNK,
    'derivative_chunk': DERIVATIVE_CHUNK,
    'coordinate_pairs': [list(pair) for pair in COORDINATE_PAIRS],
    'plot_3d_columns': PLOT_3D_COLUMNS,
    'plot_3d_max_points': PLOT_3D_MAX_POINTS,
    'plot_3d_elevation': PLOT_3D_ELEV,
    'plot_3d_azimuth': PLOT_3D_AZIM,
    'plot_3d_point_size': PLOT_3D_POINT_SIZE,
    'plot_3d_alpha': PLOT_3D_ALPHA,
    'equilibrium_labels': EQUILIBRIUM_LABELS,
    'equilibria': KNOWN_EQUILIBRIA.detach().cpu().tolist(),
    'seed': SEED,
    'em_seed': EM_SEED,
    'device': str(DEVICE),
    'dtype': str(DTYPE),
    'save_run': SAVE_RUN,
    'output_root': str(EXPERIMENT_DIR / 'saved_runs'),
}

# Print a complete, readable statement immediately after initialization.
print('=' * 76)
print(f'STOCHASTIC {DIM}D COURNOT NEURAL-DTB EXPERIMENT')
print('=' * 76)
print(f'Game/player dimension: d={DIM}; b={COURNOT_B}; mu={COURNOT_MU}')
print(f'Particles and time: N={N_PARTICLES}; h={H}; K={N_STEPS}; T={T_FINAL}')
print(f'Snapshot times: {SNAPSHOT_TIMES}')
print(f'Noise amplitudes sigma_i: {NOISE_STD}')
print(f'Diffusion covariance diagonal D_ii: {CONFIG["diffusion_diagonal"]}')
print(f'Maximum equilibrium drift residual: {equilibrium_residual:.3e}')
print(f'Initial law: {INITIAL_LAW}; Gaussian mean/std={GAUSSIAN_MEAN}/{GAUSSIAN_STD}; smoothing={UNIFORM_SMOOTHING_STD}')
print(f'Initial score RMS: {float(q_0.square().sum(dim=1).mean().sqrt()):.6g}')
print(f'Network: {NN_CHOICE}; {DIM} -> ' + ' -> '.join([str(MLP_WIDTH)] * MLP_DEPTH) + f' -> {DIM}')
print(f'Activation/depth/width: {NN_ACTIVATION}/{MLP_DEPTH}/{MLP_WIDTH}')
print(f'Trainable parameters: {N_PARAMETERS}')
print(f'Selected tangent size: {BASIS_SIZE}; selection seed={SEED}')
print(f'Selected flat-parameter indices: {CONFIG["selected_parameter_indices"]}')
print(f'Projection solver: {SVD_METHOD}; relative SVD tolerance={SVD_RTOL}')
print(f'Jacobian/derivative chunks: {JACOBIAN_CHUNK}/{DERIVATIVE_CHUNK}')
print(f'Device/dtype: {DEVICE}/{DTYPE}; EM seed={EM_SEED}')
print(f'Coordinate planes: {COORDINATE_PAIRS}')
print(f'Save run: {SAVE_RUN}; output root={CONFIG["output_root"]}')
print('Reported reference equilibria:')
for index, point in enumerate(KNOWN_EQUILIBRIA.detach().cpu().numpy()):
    stability = 'stable' if STABLE_MASK[index] else 'unstable'
    print(f'  E_{index} = {np.array2string(point, precision=8)} [{stability}]')
if INITIAL_LAW == 'uniform':
    print('Uniform-score note: q_0=0 is the exact interior score; the boundary score is singular.')
print('=' * 76)

## Block 3 — Stochastic DTB tangent and score update

At step $k$, the Fokker--Planck transport target is

$$
g_{k,i}=b(x_{k,i})-\frac12Dq_{k,i}.
$$

The selected neural tangent basis is projected onto this target:

$$
\alpha_k=\arg\min_\alpha\sum_i\|J_{k,i}^S\alpha-g_{k,i}\|_2^2,
\qquad u_k(x)=\partial_{\theta_S}f_{\theta_0}(x)\alpha_k.
$$

Particles and attached scores use the same old-step velocity:

$$
x_i^{k+1}=x_i^k+h u_k(x_i^k),
$$

$$
q_i^{k+1}=q_i^k-h\left([D_xu_k(x_i^k)]^\mathsf{T}q_i^k
+\nabla_x[\nabla_x\!\cdot u_k](x_i^k)\right).
$$

This is the ordinary fixed-basis DTB update: there is no resampling, refit,
score network, or post-run polish step.
The matched EM baseline uses independent Gaussian increments:

$$
x_{k+1}^{\mathrm{EM}}=x_k^{\mathrm{EM}}+h b(x_k^{\mathrm{EM}})
+\sqrt{h}\,\operatorname{diag}(\sigma)\xi_k,
\qquad \xi_k\sim N(0,I_3).
$$

**Timing scope.** Separate synchronized wall-clock timers measure each method's
numerical step. DTB includes the tangent basis, SVD solve, spatial derivatives,
score evolution, and particle/log-density updates. EM includes drift evaluation,
Gaussian noise generation, and the particle update. Initialization, diagnostics,
CPU history copies, printing, plotting, and saving are excluded from both method
timers. The first step is included (no warm-up); this is a single-run comparison
at the stated particle count and time step, not an equal-accuracy benchmark.
The combined loop wall time, which also includes diagnostics and history storage,
is reported separately. CUDA/MPS work is synchronized at timer boundaries.


In [ ]:
# BLOCK 3 — Evolve and separately time DTB and Euler--Maruyama.
state_times = np.arange(N_STEPS + 1, dtype=float) * H
solve_times = np.arange(N_STEPS, dtype=float) * H

# Histories are copied to CPU after each step to keep GPU memory bounded.
dtb_history = [x_k.detach().cpu().numpy()]
score_history = [q_k.detach().cpu().numpy()]
log_density_history = [log_density_k.detach().cpu().numpy()]
em_history = [em_k.detach().cpu().numpy()]
coefficient_history = []
diagnostic_records = []

def synchronize_device():
    """Finish queued accelerator work so wall timers measure actual execution."""
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)
    elif DEVICE.type == 'mps':
        torch.mps.synchronize()


dtb_step_seconds = np.empty(N_STEPS, dtype=float)
em_step_seconds = np.empty(N_STEPS, dtype=float)
synchronize_device()
run_start = time.perf_counter()
for step in range(N_STEPS):
    step_start = time.perf_counter()
    synchronize_device()
    dtb_start = time.perf_counter()

    # 1. Evaluate the game drift and score-dependent diffusion correction.
    drift_k = game_velocity(x_k)
    diffusion_correction = 0.5 * (q_k @ diffusion.T)
    target_k = drift_k - diffusion_correction

    # 2. Build the selected neural tangent basis at the current physical particles.
    _, jacobian_tensor, stacked_jacobian = game_dtb_basis_matrices(
        theta_0,
        selected,
        x_k,
        mlp,
        structure,
        chunk=JACOBIAN_CHUNK,
    )

    # 3. Project the stochastic target velocity using the shared DTB SVD solver.
    alpha_k = jform_solve(
        stacked_jacobian,
        target_k.reshape(-1),
        rtol=SVD_RTOL,
        method=SVD_METHOD,
    )

    # 4. Differentiate only the final projected direction u_k=J_k alpha_k.
    tangent_k, grad_u_k, divergence_k, grad_divergence_k = tangent_velocity_spatial_terms(
        theta_0,
        selected,
        alpha_k,
        x_k,
        mlp,
        structure,
        chunk_size=DERIVATIVE_CHUNK,
    )

    # 5. Update the score with (D_x u_k)^T q_k and grad(div(u_k)).
    q_next, transported_score, score_source = euler_score_update(
        q_k,
        grad_u_k,
        grad_divergence_k,
        H,
    )

    # 6. Apply the ordinary DTB particle-map Euler step using the same old u_k.
    x_next = x_k + H * tangent_k
    log_density_next = log_density_k - H * divergence_k

    synchronize_device()
    dtb_step_seconds[step] = time.perf_counter() - dtb_start

    # 7. Time the complete EM step, including its independent Gaussian noise.
    synchronize_device()
    em_start = time.perf_counter()
    em_noise = torch.randn(
        em_k.shape, device=DEVICE, dtype=DTYPE, generator=em_generator,
    )
    em_next = em_k + H * game_velocity(em_k) + math.sqrt(H) * sigma * em_noise
    synchronize_device()
    em_step_seconds[step] = time.perf_counter() - em_start

    # 8. Compute diagnostics outside both method timers, using old-step states.
    residual_k = tangent_k - target_k
    target_norm = target_k.norm().clamp_min(1e-30)
    relative_error = residual_k.norm() / target_norm
    particle_scale = math.sqrt(N_PARTICLES)
    diagnostic_records.append({
        'time': step * H,
        'relative_projection_error': float(relative_error),
        'alpha_norm': float(alpha_k.norm()),
        'score_rms': float(q_k.norm() / particle_scale),
        'diffusion_rms': float(diffusion_correction.norm() / particle_scale),
        'target_rms': float(target_k.norm() / particle_scale),
        'tangent_rms': float(tangent_k.norm() / particle_scale),
        'score_transport_rms': float(transported_score.norm() / particle_scale),
        'score_source_rms': float(score_source.norm() / particle_scale),
        'dtb_compute_seconds': dtb_step_seconds[step],
        'em_compute_seconds': em_step_seconds[step],
        'seconds': time.perf_counter() - step_start,
    })
    coefficient_history.append(alpha_k.detach().cpu().numpy())

    # 9. Detach the new state because no gradient graph is propagated across time.
    x_k = x_next.detach()
    q_k = q_next.detach()
    log_density_k = log_density_next.detach()
    em_k = em_next.detach()

    # 10. Store full histories so any requested time can be visualized afterward.
    dtb_history.append(x_k.cpu().numpy())
    score_history.append(q_k.cpu().numpy())
    log_density_history.append(log_density_k.cpu().numpy())
    em_history.append(em_k.cpu().numpy())

    # 11. Fail immediately and identify any non-finite DTB component.
    dtb_states = {
        'particles': x_k,
        'score': q_k,
        'log_density': log_density_k,
    }
    nonfinite_states = [
        name for name, value in dtb_states.items() if not bool(torch.isfinite(value).all())
    ]
    if nonfinite_states:
        raise FloatingPointError(
            f'Non-finite DTB state after step {step + 1}: {nonfinite_states}.'
        )
    if not bool(torch.isfinite(em_k).all()):
        raise FloatingPointError(f'Non-finite EM particles after step {step + 1}.')
    if (step + 1) % PRINT_EVERY == 0 or step in (0, N_STEPS - 1):
        record = diagnostic_records[-1]
        print(
            f'k={step + 1:4d}/{N_STEPS}  t={(step + 1) * H:.4f}  '
            f'rel.err={record["relative_projection_error"]:.3e}  '
            f'||alpha||={record["alpha_norm"]:.3e}  '
            f'score.rms={record["score_rms"]:.3e}  '
            f'DTB={dtb_step_seconds[step]:.4f}s  EM={em_step_seconds[step]:.6f}s'
        )

dtb_history = np.stack(dtb_history)
score_history = np.stack(score_history)
log_density_history = np.stack(log_density_history)
em_history = np.stack(em_history)
coefficient_history = np.stack(coefficient_history)
synchronize_device()
CONFIG['wall_seconds'] = time.perf_counter() - run_start
CONFIG['timing_scope'] = (
    'Synchronized numerical steps, including first-step overhead; excludes '
    'initialization, diagnostics, history copies, printing, plotting, and saving.'
)
CONFIG['dtb_compute_seconds'] = float(dtb_step_seconds.sum())
CONFIG['em_compute_seconds'] = float(em_step_seconds.sum())
CONFIG['dtb_mean_step_seconds'] = float(dtb_step_seconds.mean())
CONFIG['em_mean_step_seconds'] = float(em_step_seconds.mean())
CONFIG['dtb_over_em_time_ratio'] = (
    CONFIG['dtb_compute_seconds'] / CONFIG['em_compute_seconds']
)
timing_table = '\n'.join([
    '| Method | Computation total (s) | Mean step (ms) |',
    '| :--- | ---: | ---: |',
    f'| Neural-DTB | {CONFIG["dtb_compute_seconds"]:.6f} | {1000 * CONFIG["dtb_mean_step_seconds"]:.6f} |',
    f'| Euler--Maruyama | {CONFIG["em_compute_seconds"]:.6f} | {1000 * CONFIG["em_mean_step_seconds"]:.6f} |',
])
display(Markdown('## DTB versus EM computation time'))
display(Markdown(timing_table))
print(f'DTB / EM computation-time ratio: {CONFIG["dtb_over_em_time_ratio"]:.2f}x')
print(f'Combined loop wall time (both methods + diagnostics/storage): {CONFIG["wall_seconds"]:.2f} s')
print(CONFIG['timing_scope'])

## Block 4 — Matched 3D particle clouds

Each method gets a separate snapshot grid at `SNAPSHOT_TIMES` (by default
$t=0,0.2,0.4,0.6,0.8,1$). Every panel uses the same camera and cubic axis
limits across both methods, with no clipping to $[0,1]^3$. Red diamonds and
labels $E_0,\ldots,E_4$ identify the deterministic drift equilibria; each
figure includes their coordinates. These markers do not assert stability.

`PLOT_3D_MAX_POINTS=None` displays every particle. A smaller value uses the
same seeded subset for all panels and only changes the display, not the run.
With `SAVE_RUN=True`, both figures are saved as PNG files.


In [ ]:
# BLOCK 4 — Show separate 3D cloud figures with shared axes and labeled equilibria.
output_dir = None
if SAVE_RUN:
    output_dir = create_run_directory(
        EXPERIMENT_DIR / 'saved_runs',
        dim=DIM,
        network=NN_CHOICE,
        activation=NN_ACTIVATION,
        width=MLP_WIDTH,
        depth=MLP_DEPTH,
        parameter_count=N_PARAMETERS,
        basis_size=BASIS_SIZE,
        seed=SEED,
    )
    CONFIG['output_dir'] = str(output_dir)

def plot_3d_cloud_snapshots(dtb, em, times, steps, equilibria):
    """Return one matched snapshot grid per method, without clipping particle data."""
    dtb, em = np.asarray(dtb), np.asarray(em)
    times, steps = np.asarray(times), np.asarray(steps, dtype=int)
    references = np.asarray(equilibria)
    if dtb.shape != em.shape or dtb.ndim != 3 or dtb.shape[-1] != 3 or dtb.shape[1] == 0:
        raise ValueError('Both histories must have the same nonempty (time, particles, 3) shape.')
    if times.shape != (dtb.shape[0],):
        raise ValueError('times must have one value per stored state.')
    if steps.ndim != 1 or steps.size == 0 or np.any(steps < 0) or np.any(steps >= len(times)):
        raise ValueError('Snapshot indices must be nonempty and lie in the stored history.')
    if references.shape != (len(EQUILIBRIUM_LABELS), 3):
        raise ValueError('Each equilibrium needs three coordinates and one label.')
    if not all(np.isfinite(value).all() for value in (dtb, em, times, references)):
        raise ValueError('Cannot plot non-finite histories, times, or equilibria.')
    if not isinstance(PLOT_3D_COLUMNS, int) or PLOT_3D_COLUMNS < 1:
        raise ValueError('PLOT_3D_COLUMNS must be a positive integer.')
    if PLOT_3D_MAX_POINTS is not None and (
        not isinstance(PLOT_3D_MAX_POINTS, int) or PLOT_3D_MAX_POINTS < 1
    ):
        raise ValueError('PLOT_3D_MAX_POINTS must be None or a positive integer.')

    # Select once, using a local RNG so visualization never changes simulation RNGs.
    indices = np.arange(dtb.shape[1])
    if PLOT_3D_MAX_POINTS is not None and PLOT_3D_MAX_POINTS < len(indices):
        indices = np.sort(np.random.default_rng(SEED).choice(
            indices, size=PLOT_3D_MAX_POINTS, replace=False,
        ))

    # All points at all displayed times contribute to the common cubic bounds,
    # including Brownian excursions beyond [0,1]^3 and any omitted display points.
    shown_values = np.concatenate((dtb[steps].ravel(), em[steps].ravel(), references.ravel()))
    low, high = float(shown_values.min()), float(shown_values.max())
    padding = 0.08 * max(high - low, 1e-3)
    limits = (low - padding, high + padding)
    label_offset = 0.025 * (limits[1] - limits[0])
    columns = min(PLOT_3D_COLUMNS, len(steps))
    rows = math.ceil(len(steps) / columns)
    figures = {}
    for name, history, color in (
        ('dtb', dtb, '#2389bd'), ('em', em, '#2389bd'),
    ):
        method = 'Neural-DTB' if name == 'dtb' else 'Euler--Maruyama'
        fig = plt.figure(figsize=(5.4 * columns, 4.2 * rows + 1.25))
        for panel, step in enumerate(steps, start=1):
            ax = fig.add_subplot(rows, columns, panel, projection='3d', computed_zorder=False)
            points = history[step, indices]
            ax.scatter(
                points[:, 0], points[:, 1], points[:, 2],
                s=PLOT_3D_POINT_SIZE, alpha=PLOT_3D_ALPHA, color=color,
                edgecolors='none', depthshade=False, rasterized=True, zorder=2,
            )
            ax.scatter(
                references[:, 0], references[:, 1], references[:, 2],
                s=40, c='#c62828', marker='D', edgecolors='white',
                linewidths=0.6, depthshade=False, zorder=4,
            )
            for index, point in enumerate(references):
                ax.text(
                    *(point + label_offset), rf'$E_{{{index}}}$',
                    color='#a51f1f', fontsize=9, fontweight='bold', zorder=5,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=0.6),
                )
            ax.set(
                xlim=limits, ylim=limits, zlim=limits,
                xlabel=r'$x_1$', ylabel=r'$x_2$', zlabel=r'$x_3$',
                title=rf'$t={times[step]:.3g}$',
            )
            ax.set_box_aspect((1, 1, 1))
            ax.view_init(elev=PLOT_3D_ELEV, azim=PLOT_3D_AZIM)
            ax.tick_params(labelsize=8, pad=1)
            for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
                axis.pane.fill = False
            ax.grid(True, alpha=0.2)
        fig.suptitle(f'{method} — 3D particle clouds', fontsize=15, y=0.985)
        fig.text(
            0.5, 0.957,
            f'{len(indices):,} of {dtb.shape[1]:,} particles per snapshot · matched view and axis limits',
            ha='center', fontsize=10, color='#444444',
        )
        # A coordinate key keeps the labels short enough to read inside every panel.
        key = [
            rf'$E_{{{i}}}=({", ".join(f"{value:.3g}" for value in point)})$'
            for i, point in enumerate(references)
        ]
        fig.text(0.5, 0.047, 'Deterministic drift equilibria (red diamonds)', ha='center', fontsize=10)
        fig.text(0.5, 0.025, '   '.join(key[:3]) + '\n' + '   '.join(key[3:]),
                 ha='center', va='center', fontsize=10, linespacing=1.5)
        fig.subplots_adjust(left=0.04, right=0.96, bottom=0.085, top=0.925,
                            wspace=0.08, hspace=0.15)
        figures[name] = fig
    return figures


cloud_3d_figures = plot_3d_cloud_snapshots(
    dtb_history, em_history, state_times, snapshot_steps,
    KNOWN_EQUILIBRIA.detach().cpu().numpy(),
)
if output_dir is not None:
    for name, figure in cloud_3d_figures.items():
        figure.savefig(output_dir / f'{name}_3d_cloud_snapshots.png', dpi=180, bbox_inches='tight')
plt.show()


## Block 5 — DTB snapshots and diagnostics

The snapshot columns show the neural-DTB particles at the requested
physical times with shared axis limits. The tangent figure shows the
relative projection error and $\|\alpha_k\|_2$ across time. No PCA
is used.
The timing table and per-step timings are saved with the configuration and histories
when `SAVE_RUN=True`. The separate 3D figures above show both methods.


In [ ]:
# BLOCK 5 — Keep the DTB coordinate plots/diagnostics, then optionally save both methods.
# Plot only the DTB cloud, using common limits across every requested time.
def finite_plot_limits(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return (-1.0, 1.0)
    low, high = float(values.min()), float(values.max())
    padding = 0.06 * max(high - low, 1e-3)
    return low - padding, high + padding

snapshot_figures = {}
references = KNOWN_EQUILIBRIA.detach().cpu().numpy()
for first, second in COORDINATE_PAIRS:
    if first == second or not (1 <= first <= DIM and 1 <= second <= DIM):
        raise ValueError('Coordinate pairs use distinct one-based indices.')
    a, b = first - 1, second - 1
    xlim = finite_plot_limits(np.concatenate((dtb_history[:, :, a].ravel(), references[:, a])))
    ylim = finite_plot_limits(np.concatenate((dtb_history[:, :, b].ravel(), references[:, b])))
    figure, axes = plt.subplots(
        1,
        len(snapshot_steps),
        squeeze=False,
        figsize=(3.0 * len(snapshot_steps), 3.4),
        constrained_layout=True,
    )
    for column, snapshot_step in enumerate(snapshot_steps):
        axis = axes[0, column]
        axis.scatter(
            dtb_history[snapshot_step, :, a],
            dtb_history[snapshot_step, :, b],
            s=5, alpha=0.38, color='#2389bd', edgecolors='none', rasterized=True,
        )
        if np.any(STABLE_MASK):
            axis.scatter(
                references[STABLE_MASK, a], references[STABLE_MASK, b],
                s=30, color='#d43e38', marker='D', edgecolors='white',
                linewidths=0.4, label='stable reference', zorder=4,
            )
        if np.any(~STABLE_MASK):
            axis.scatter(
                references[~STABLE_MASK, a], references[~STABLE_MASK, b],
                s=42, color='#ffbf00', marker='X', edgecolors='#333333',
                linewidths=0.5, label='unstable reference', zorder=4,
            )
        axis.set(
            xlim=xlim, ylim=ylim, xlabel=rf'$x_{{{first}}}$',
            ylabel=rf'$x_{{{second}}}$', title=rf'$t={state_times[snapshot_step]:.3g}$',
        )
        axis.set_aspect('equal', adjustable='box')
        axis.grid(alpha=0.25)
    handles, labels = axes[0, -1].get_legend_handles_labels()
    if handles:
        figure.legend(handles, labels, loc='upper right', fontsize=8)
    figure.suptitle(rf'Neural-DTB snapshots on the $(x_{{{first}}},x_{{{second}}})$ plane')
    if output_dir is not None:
        figure.savefig(
            output_dir / f'dtb_snapshots_x{first}_x{second}.png',
            dpi=180, bbox_inches='tight',
        )
    snapshot_figures[(first, second)] = figure

# Plot exactly the two requested tangent-bundle diagnostics against physical time.
relative_projection_errors = np.asarray([
    record['relative_projection_error'] for record in diagnostic_records
])
alpha_norms = np.asarray([record['alpha_norm'] for record in diagnostic_records])
diagnostics_figure = plot_tangent_diagnostics(
    solve_times,
    relative_projection_errors,
    alpha_norms,
    output_path=None if output_dir is None else output_dir / 'tangent_diagnostics.png',
)
plt.show()

# Display the diagnostic definitions and their first, mean, and final values in math.
metric_table = diagnostics_markdown(diagnostic_records)
display(Markdown('## Stochastic DTB diagnostics'))
display(Markdown(metric_table))

# SAVE_RUN=False leaves the repository unchanged and only displays results.
if output_dir is not None:
    save_run_data(
        output_dir,
        config=CONFIG,
        arrays={
            'times': state_times,
            'solve_times': solve_times,
            'dtb_particles': dtb_history,
            'em_particles': em_history,
            'dtb_step_seconds': dtb_step_seconds,
            'em_step_seconds': em_step_seconds,
            'scores': score_history,
            'log_density': log_density_history,
            'coefficients': coefficient_history,
            'selected_parameter_indices': selected.detach().cpu().numpy(),
            'relative_projection_error': relative_projection_errors,
            'alpha_norm': alpha_norms,
        },
        metrics_markdown=timing_table + '\n\n' + CONFIG['timing_scope'] + '\n\n' + metric_table,
    )
    print('Saved figures, histories, configuration, and metric tables to:', output_dir)
else:
    print('SAVE_RUN=False: results were displayed without creating an output folder.')